# doc-extraction — OmniDocBench benchmark (Kaggle, T4 GPU)

Thin orchestrator notebook — no extraction or evaluation logic lives
here. Everything runs through the repo's own scripts
(`experiments/005_omnidocbench/run.py`,
`src/doc_extraction/evaluation/omnidocbench.py`). See
[`experiments/005_omnidocbench/README.md`](README.md) for the full design,
the pinned OmniDocBench commit, and what has/hasn't been run locally.

**No private data.** Only the public OmniDocBench dataset is used here.
This repo's own `data/` (local/private sample documents) is gitignored
and is not part of this clone — never attach it as a Kaggle input.

**Two separate Python environments, on purpose:**

```
Kaggle kernel (Python 3.12)
    doc-extraction pipeline (prediction generation)
        |
        v  subprocess
Isolated venv (Python 3.11, built with uv)
    official OmniDocBench evaluator (scoring)
```

The pinned evaluator requires Python `>=3.10,<3.12`; Kaggle's kernel is
3.12+. The evaluator always runs as an external subprocess in its own
venv — it is never imported into the main kernel, its source is never
modified, and nothing is monkey-patched onto `sys.path`.

**CPU pipeline vs. GPU vs. evaluator** — do not conflate these:
- The extraction pipeline (`baseline`, `docling` backends) runs on **CPU**
  by default (`configs/cpu.yaml`, `device: cpu`). Neither backend
  currently requests CUDA — this notebook prints GPU info for
  diagnostics only and does not claim the T4 accelerates the current run.
- The OmniDocBench evaluator is CPU-only Python `<3.12` and never touches
  the GPU either way.

In [ ]:
print("==================================================")
print("1. RUNTIME")
print("==================================================")

Python version and GPU diagnostics. Diagnostic only — see the note
above on why a visible T4 does not by itself mean the pipeline uses it.

In [ ]:
import sys
print("Kernel Python:", sys.version)
assert sys.version_info >= (3, 10), "doc_extraction needs Python 3.10+"

In [ ]:
import torch

has_cuda = torch.cuda.is_available()
print(f"CUDA available: {has_cuda}")
if has_cuda:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA: {torch.version.cuda}")
    print(f"VRAM GB: {round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)}")
else:
    print("No GPU visible — check Settings > Accelerator is set to GPU T4 x2 (or T4 x1).")

print()
print("Reminder: the baseline/docling backends run on CPU regardless of the")
print("above (configs/cpu.yaml, device: cpu) — see the intro cell.")

In [ ]:
print("==================================================")
print("2. REPOSITORY")
print("==================================================")

Idempotent: reruns of this cell reuse an existing clone instead of
failing on `git clone` into a non-empty directory, and pull the latest
commit so a stale clone from an earlier session doesn't shadow fixes.

In [ ]:
import os

REPO_DIR = "/kaggle/working/doc-extraction"

if not os.path.exists(REPO_DIR):
    os.chdir("/kaggle/working")
    !git clone https://github.com/anhkhoa1804/doc-extraction.git
else:
    print("Repository already exists:", REPO_DIR, "- pulling latest")
    !git -C {REPO_DIR} pull --ff-only

os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!git rev-parse HEAD

In [ ]:
print("==================================================")
print("3. DOC-EXTRACTION ENVIRONMENT")
print("==================================================")

The main project, installed into the Kaggle kernel's own Python 3.12 —
no isolated venv needed here, its dependencies have no `<3.12` ceiling.

In [ ]:
%pip install -e ".[docling,tables]"

In [ ]:
import doc_extraction
print("doc_extraction imported OK:", doc_extraction.__file__)

**Prefetch Docling models — required on Kaggle, not optional.**
`docs/setup.md` calls this optional for the local dev machine, but only
because its cache already had models from prior local runs. Once
`DOCLING_ARTIFACTS_PATH` is set to anything, Docling refuses to
auto-download and requires that directory to already contain the
models — it raises `RuntimeError: ... is not valid` otherwise on a
fresh session. Prefetch once, up front.

In [ ]:
import os

CACHE_ROOT = "/kaggle/working/.cache"
os.makedirs(f"{CACHE_ROOT}/huggingface", exist_ok=True)
os.makedirs(f"{CACHE_ROOT}/docling", exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_ROOT}/huggingface"
os.environ["DOCLING_ARTIFACTS_PATH"] = f"{CACHE_ROOT}/docling"
os.environ["XDG_CACHE_HOME"] = CACHE_ROOT

# Default model set (layout, table structure, ...) plus EasyOCR for the
# en/vi languages configs/cpu.yaml is configured for. Note: OmniDocBench
# also contains Chinese-language pages, which these EasyOCR language
# models do not cover — a pre-existing config choice inherited from this
# repo's own corpus, not something this notebook changes.
!python -m docling.cli.tools models download -o {CACHE_ROOT}/docling
!python -m docling.cli.tools models download easyocr \
    --easyocr-lang en --easyocr-lang vi -o {CACHE_ROOT}/docling

In [ ]:
print("==================================================")
print("4. OMNIDOCBENCH EVALUATOR ENVIRONMENT")
print("==================================================")

This is the section that was broken: the evaluator's own
`pyproject.toml` (at the pinned commit) declares `PyYAML==6.0.2` as a
normal dependency, so `pip install -e <evaluator>` should install it —
yet the evaluator interpreter still reported
`ModuleNotFoundError: No module named 'yaml'`. That mismatch ("uv says
it's installed, the interpreter disagrees") means the *specific*
interpreter used at runtime was not verified to be the one the install
actually targeted. The fix here is a health check that imports `yaml`
**inside a subprocess run with the exact absolute path** `run.py` will
later use, plus a clean-rebuild path if that check fails — not another
one-off `pip install PyYAML` patch.

In [ ]:
import os

OMNIDOC_REPO = f"{REPO_DIR}/.external/OmniDocBench"
PINNED_OMNIDOC_COMMIT = "193627ae9e97d89188468ed1ee3b7a856ff76044"

if not os.path.exists(OMNIDOC_REPO):
    os.makedirs(f"{REPO_DIR}/.external", exist_ok=True)
    !git clone https://github.com/opendatalab/OmniDocBench.git {OMNIDOC_REPO}
!git -C {OMNIDOC_REPO} fetch --all --tags -q
!git -C {OMNIDOC_REPO} checkout {PINNED_OMNIDOC_COMMIT}

import subprocess
_actual_commit = subprocess.run(
    ["git", "-C", OMNIDOC_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True, check=True,
).stdout.strip()
assert _actual_commit == PINNED_OMNIDOC_COMMIT, (
    f"OmniDocBench checkout is at {_actual_commit}, expected pinned {PINNED_OMNIDOC_COMMIT}"
)
print(f"OmniDocBench checked out at pinned commit {PINNED_OMNIDOC_COMMIT}")

In [ ]:
# Absolute paths only, throughout — a relative '.venv-omnidoc/bin/python'
# silently resolves to a different (wrong or nonexistent) location the
# moment any cell's cwd differs, which is exactly the kind of mismatch
# that produces "uv says it's there, the interpreter says otherwise".
OMNIDOC_VENV = f"{REPO_DIR}/.venv-omnidoc"
OMNIDOC_PYTHON = f"{OMNIDOC_VENV}/bin/python"

print("OMNIDOC_VENV:", OMNIDOC_VENV)
print("OMNIDOC_PYTHON:", OMNIDOC_PYTHON)

In [ ]:
import json
import os
import shutil
import subprocess


def check_evaluator_health():
    """Runs *inside* OMNIDOC_PYTHON via subprocess — the only way to be
    sure we're checking what run.py will actually invoke, not what `uv`
    or `pip` merely claim about some environment."""
    if not os.path.exists(OMNIDOC_PYTHON):
        return False, f"no interpreter at {OMNIDOC_PYTHON}"

    probe = (
        "import sys, json; "
        "info = {'executable': sys.executable, 'version': list(sys.version_info[:2])}; "
        "import yaml; info['yaml_file'] = yaml.__file__; info['yaml_version'] = yaml.__version__; "
        "from src.core.pipeline import run_config_file; info['entrypoint'] = 'OK'; "
        "print(json.dumps(info))"
    )
    result = subprocess.run(
        [OMNIDOC_PYTHON, "-c", probe],
        cwd=OMNIDOC_REPO,
        capture_output=True, text=True,
    )
    if result.returncode != 0:
        tail = (result.stderr or result.stdout).strip().splitlines()[-15:]
        return False, "\n".join(tail)

    try:
        info = json.loads(result.stdout.strip().splitlines()[-1])
    except (ValueError, IndexError) as exc:
        return False, f"could not parse health-check output: {exc}; raw={result.stdout!r}"

    major, minor = info["version"]
    if not (major == 3 and 10 <= minor < 12):
        return False, f"interpreter is Python {major}.{minor}, need 3.10 or 3.11 (<3.12)"
    if not os.path.realpath(info["executable"]).startswith(os.path.realpath(OMNIDOC_VENV)):
        return False, f"sys.executable={info['executable']!r} is not under {OMNIDOC_VENV}"

    print(f"  interpreter : {info['executable']}")
    print(f"  version     : Python {major}.{minor}")
    print(f"  yaml        : {info['yaml_version']} ({info['yaml_file']})")
    print(f"  entrypoint  : {info['entrypoint']}")
    return True, info


def build_evaluator_venv():
    print(f"Building evaluator venv at {OMNIDOC_VENV} ...")
    if os.path.exists(OMNIDOC_VENV):
        shutil.rmtree(OMNIDOC_VENV)
    subprocess.run(["uv", "python", "install", "3.11"], check=True)
    subprocess.run(["uv", "venv", "--python", "3.11", OMNIDOC_VENV], check=True)
    subprocess.run(
        ["uv", "pip", "install", "--python", OMNIDOC_PYTHON, "-U", "pip", "setuptools", "wheel", "-q"],
        check=True,
    )
    # Installs the evaluator AND every dependency declared in its own
    # pyproject.toml (PyYAML included) — the upstream declaration is the
    # source of truth, not a hand-picked subset.
    subprocess.run(
        ["uv", "pip", "install", "--python", OMNIDOC_PYTHON, "-e", OMNIDOC_REPO, "-q"],
        check=True,
    )

In [ ]:
!pip install -q -U uv

print("Checking for an existing, healthy evaluator venv...")
ok, info = check_evaluator_health()

if not ok:
    print(f"Not healthy ({info}) — building/rebuilding from scratch.")
    build_evaluator_venv()
    ok, info = check_evaluator_health()

assert ok, f"Evaluator venv still unhealthy after a clean rebuild:\n{info}"
print("\nEvaluator environment OK.")

In [ ]:
print("==================================================")
print("5. DATASET")
print("==================================================")

**Option A (the real benchmark): attach a Kaggle Dataset** containing
`OmniDocBench.json` + `images/` (from
https://huggingface.co/datasets/opendatalab/OmniDocBench) via "Add
Input" — it appears at `/kaggle/input/<dataset-name>/`. Edit
`KAGGLE_DATASET_NAME` below to match.

**Option B (smoke/demo testing only): the bundled 18-page official demo
set.** This is *never* silently treated as "the full benchmark" — the
`USING_FULL_DATASET` flag below is `False` whenever it's in use, and the
full-benchmark cell later in this notebook refuses to run while it is.

Either way, only public OmniDocBench data — never this repo's own `data/`.

In [ ]:
import os

# Edit this to the Kaggle Dataset you attached via "Add Input".
KAGGLE_DATASET_NAME = "<dataset-name>"

_attached_path = f"/kaggle/input/{KAGGLE_DATASET_NAME}"
_demo_path = f"{OMNIDOC_REPO}/demo_data/omnidocbench_demo"

USING_FULL_DATASET = os.path.exists(_attached_path)

if USING_FULL_DATASET:
    DATASET_PATH = _attached_path
    print(f"Attached Kaggle Dataset found — using the FULL dataset.")
else:
    DATASET_PATH = _demo_path
    print(f"No Kaggle Dataset attached at {_attached_path}.")
    print(f"Using the 18-page OFFICIAL DEMO SET for smoke/demo testing only.")
    print(f"The full-benchmark cell below will refuse to run until a real")
    print(f"dataset is attached and KAGGLE_DATASET_NAME is set to match it.")

OUTPUT_ROOT = "/kaggle/working/results"

print()
print(f"USING_FULL_DATASET = {USING_FULL_DATASET}")
print(f"DATASET_PATH       = {DATASET_PATH}")

In [ ]:
assert os.path.exists(DATASET_PATH), (
    f"DATASET_PATH does not exist: {DATASET_PATH} — if this is the demo-set "
    f"fallback, re-run the OMNIDOCBENCH EVALUATOR ENVIRONMENT section first "
    f"(it clones .external/OmniDocBench, which the demo set lives inside)."
)
print(f"DATASET_PATH = {DATASET_PATH}")
print(sorted(os.listdir(DATASET_PATH))[:20])

In [ ]:
print("==================================================")
print("6. EXPERIMENT CLI")
print("==================================================")

The actual current CLI surface `run.py` exposes — printed rather than
assumed, so every command below matches what the flags really are.

In [ ]:
!python experiments/005_omnidocbench/run.py --help

In [ ]:
print("==================================================")
print("7. SMOKE TEST")
print("==================================================")

Predictions only (`--skip-evaluate`) for a small, deterministic
3-page subset — this is the step that already proved the extraction
pipeline itself works end-to-end on Kaggle (3/3 pages ok, ~190s wall
clock on CPU; slow is expected, see the README, not a failure to fix
here). Scoring those predictions is the next section, kept separate so
a failure there doesn't force regenerating predictions to retry.

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --subset 3 \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --skip-evaluate

In [ ]:
print("==================================================")
print("8. EVALUATION")
print("==================================================")

Scores the predictions from the smoke test using `--skip-prepare` —
this reuses `/kaggle/working/results/baseline_smoke/predictions/` as-is rather
than regenerating them. **If this cell fails, fix the evaluator
environment (rerun the OMNIDOCBENCH EVALUATOR ENVIRONMENT section
above, which will rebuild it) and rerun this exact cell — do not rerun
the smoke test cell first.** CDM is deliberately left disabled
(`--include-cdm` omitted): it needs a Linux TeX Live + ImageMagick +
Ghostscript toolchain this environment hasn't been confirmed to have —
see `experiments/005_omnidocbench/README.md`. The metrics below are
whatever the evaluator actually computed without it (text edit
distance, table TEDS, reading order, ...), not a fabricated overall
score.

In [ ]:
!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_smoke \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --skip-prepare

In [ ]:
print("==================================================")
print("9. 18-PAGE DEMO RUN (optional)")
print("==================================================")

Prepare + evaluate the full 18-page official demo set — useful as a
slightly larger correctness check than the 3-page smoke test. **This
is still not the full benchmark** regardless of what `DATASET_PATH`
resolved to above; it always targets the bundled demo set explicitly.
Skip this cell if you only care about the real benchmark and already
have a dataset attached.

In [ ]:
_demo_path = f"{OMNIDOC_REPO}/demo_data/omnidocbench_demo"

!python experiments/005_omnidocbench/run.py \
    --dataset {_demo_path} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_demo18 \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 2

In [ ]:
print("==================================================")
print("10. FULL BENCHMARK (1651 pages)")
print("==================================================")

**Guarded by `USING_FULL_DATASET`.** This cell refuses to run against
the 18-page demo set — attach the real dataset via "Add Input" and set
`KAGGLE_DATASET_NAME` in the DATASET section, then re-run from there,
before running this. `--match-workers`: keep to roughly 1/3-1/2 of the
instance's CPU count (upstream's own guidance, to avoid deadlocks/OOM
in its worker pools). At the ~50s/page observed for this pipeline's
visual route, 1651 pages is on the order of 23 hours per backend — this
cell has **not** been run end-to-end as part of this fix; see the final
report for exactly what was and wasn't executed.

In [ ]:
if not USING_FULL_DATASET:
    raise RuntimeError(
        "Refusing to run the full benchmark: no Kaggle Dataset is attached "
        "(USING_FULL_DATASET=False, DATASET_PATH is the 18-page demo set). "
        "Attach the full OmniDocBench dataset via 'Add Input', set "
        "KAGGLE_DATASET_NAME in the DATASET section above, re-run that "
        "section, then re-run this cell."
    )

!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend baseline \
    --output {OUTPUT_ROOT}/baseline_full \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 4

In [ ]:
if not USING_FULL_DATASET:
    raise RuntimeError("Refusing to run: no Kaggle Dataset is attached — see the cell above.")

!python experiments/005_omnidocbench/run.py \
    --dataset {DATASET_PATH} \
    --backend docling \
    --output {OUTPUT_ROOT}/docling_full \
    --omnidoc-python {OMNIDOC_PYTHON} \
    --match-workers 4

In [ ]:
print("==================================================")
print("11. RESULTS & METRICS")
print("==================================================")

In [ ]:
from pathlib import Path

for backend_dir in sorted(Path(OUTPUT_ROOT).glob("*")):
    report = backend_dir / "report.md"
    if report.exists():
        print(f"===== {backend_dir.name} =====")
        print(report.read_text(encoding="utf-8"))
        print()

In [ ]:
print("==================================================")
print("12. PACKAGE RESULTS")
print("==================================================")

`/kaggle/working` persists for the session and can be downloaded from
the output panel afterward. To fold results back into the repo's own
history, copy just the small committed-shape files (not `predictions/`
or the evaluator's raw debug dumps) into
`experiments/005_omnidocbench/results/<backend>/`, matching the
local-run convention described in the parent README's "Files" section.

In [ ]:
from pathlib import Path

KEEP_FILES = ("report.md", "metrics.json", "runtime.json", "run_metadata.json")

for backend_dir in sorted(Path(OUTPUT_ROOT).glob("*")):
    for name in KEEP_FILES:
        f = backend_dir / name
        if f.exists():
            print(f"kept for output panel: {f}")

print()
print("Download the files above from the Kaggle output panel, then copy")
print("them locally into experiments/005_omnidocbench/results/<backend>/.")

In [ ]:
print("==================================================")
print("13. REPRODUCIBILITY METADATA")
print("==================================================")

In [ ]:
import subprocess
from datetime import datetime, timezone

repo_commit = subprocess.run(
    ["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True,
).stdout.strip()
omnidoc_commit = subprocess.run(
    ["git", "-C", OMNIDOC_REPO, "rev-parse", "HEAD"], capture_output=True, text=True,
).stdout.strip()

print(f"Timestamp (UTC)        : {datetime.now(timezone.utc).isoformat()}")
print(f"doc-extraction commit  : {repo_commit}")
print(f"OmniDocBench commit    : {omnidoc_commit} (pinned: {PINNED_OMNIDOC_COMMIT})")
print(f"Kernel Python           : {sys.version}")
print(f"Evaluator interpreter   : {OMNIDOC_PYTHON}")
print(f"CUDA available          : {torch.cuda.is_available()}")
print(f"USING_FULL_DATASET      : {USING_FULL_DATASET}")
print(f"DATASET_PATH            : {DATASET_PATH}")
print()
print("configs/cpu.yaml (what run.py actually uses by default):")
print(Path("configs/cpu.yaml").read_text(encoding="utf-8"))